In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
df=pd.read_csv("qoute_dataset.csv")

In [7]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [8]:
df.shape

(3038, 2)

In [9]:
quotes=df["quote"]
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [10]:
quotes=quotes.str.lower()

In [11]:
import string
translator=str.maketrans('', '', string.punctuation)

quotes=quotes.apply(lambda x:x.translate(translator))

In [12]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size=10000
tokenizer=Tokenizer(vocab_size)
tokenizer.fit_on_texts(quotes)




In [37]:
sequence=tokenizer.texts_to_sequences(quotes)
word_index = tokenizer.word_index

In [15]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [16]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [17]:
x=[]
y=[]

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq=seq[:i]
        output_seq=seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [18]:
len(x)

85271

In [19]:
max_len=max(len(x) for x in x)

In [20]:
max_len

745

In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

x_padded = pad_sequences(x,
                       maxlen=max_len,        # max length of each sequence
                       padding='pre',    # 'pre' or 'post'
                        # 'pre' or 'post'
                       )

In [22]:
x_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [23]:
y=np.array(y)

In [24]:
x_padded.shape

(85271, 745)

In [25]:
y.shape

(85271,)

In [26]:
from tensorflow.keras.utils import to_categorical
y_ohe=to_categorical(y,num_classes=10000)



In [27]:
y_ohe.shape

(85271, 10000)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense



In [29]:
embedding_dim=50
rnn_units=128

In [30]:
rnn_model=Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)

rnn_model.add(
    SimpleRNN(units=rnn_units)
)
rnn_model.add(Dense(units=vocab_size,activation="softmax"))


D:\ANACONDA\ANACONDA\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [31]:
rnn_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [32]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:
lstm_model=Sequential()

lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)

lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(Dense(units=vocab_size,activation="softmax"))

In [34]:
lstm_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [35]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
epochs=2

batch_size=128

In [36]:
history_rnn=rnn_model.fit(
    x_padded,y_ohe,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,

)

Epoch 1/2
533/533 ━━━━━━━━━━━━━━━━━━━━ 43s 73ms/step - accuracy: 0.0429 - loss: 6.7476 - val_accuracy: 0.0553 - val_loss: 6.6517
Epoch 2/2
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.0688 - loss: 6.1725 - val_accuracy: 0.0808 - val_loss: 6.4983


In [37]:
history_rnn=lstm_model.fit(
    x_padded,y_ohe,
    epochs=2,
    batch_size=batch_size,
    validation_split=0.2,

)

Epoch 1/2
533/533 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.0388 - loss: 6.7584 - val_accuracy: 0.0483 - val_loss: 6.7380
Epoch 2/2
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 0.0552 - loss: 6.3350 - val_accuracy: 0.0603 - val_loss: 6.6543


In [38]:
lstm_model.save("lstm_model.h5")

In [44]:
index_to_word={}
for word,index in word_index.items():
    index_to_word[index]=word


from tensorflow.keras.preprocessing.sequence import pad_sequences

def predictor(model,tokenizer,text,max_len):
    text=text.lower()

    seq=tokenizer.texts_to_sequences([text])[0]
    seq=pad_sequences([seq],maxlen=max_len,padding="pre")

    pred=model.predict(seq)
    pred_index=np.argmax(pred)
    return index_to_word.get(pred_index,"")
    

In [51]:
seed_text="why you broke my"
next_word=predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
leftknow


In [65]:
def generate(model, tokenizer, seed_text, max_len,n_words):
    for _ in range(n_words):
        next_word = predictor(model, tokenizer, seed_text, max_len)
        if next_word == "":
            break
        seed_text += " " + next_word
    return seed_text

In [66]:
seed="the meaning of life"
generate_text=generate(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
the meaning of life loss beneath centremember centremember beneath 112 تكلف fierce down cautiously


In [67]:
import pickle
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [68]:
with open("max_len.pkl","wb") as f:
    pickle.dump(max_len,f)